In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# -----------------------------
# CONFIG
# -----------------------------
num_users = 200_000
num_items = 5
interaction_prob = 0.02   # 2% sparsity (very sparse)
min_rating = 1
max_rating = 5
seed = 42

np.random.seed(seed)

# -----------------------------
# GENERATE SPARSE DATA
# -----------------------------
rows = []
cols = []
data = []

for user in range(num_users):
    for item in range(num_items):
        if np.random.rand() < interaction_prob:
            rows.append(user)
            cols.append(item)
            rating = np.random.randint(min_rating, max_rating + 1)
            data.append(rating)

# -----------------------------
# CREATE SPARSE MATRIX A
# -----------------------------
A = csr_matrix((data, (rows, cols)), shape=(num_users, num_items))

print("Matrix shape:", A.shape)
print("Number of interactions:", A.nnz)
print("Sparsity:", 1 - (A.nnz / (num_users * num_items)))

In [ ]:
A

In [ ]:
user_ids, item_ids = A.nonzero()
ratings = A.data

# Combine into triplets
triplets = list(zip(user_ids, item_ids, ratings))

# Optional: convert to DataFrame (for easier handling)
triplet_df = pd.DataFrame(triplets, columns=["user_id", "item_id", "rating"])

# Preview
print(triplet_df.head())
print("Total triplets:", len(triplets))

In [ ]:
# -----------------------------
# STEP 3: TRAIN-TEST SPLIT
# -----------------------------

# Shuffle triplets
triplet_df = triplet_df.sample(frac=1, random_state=42).reset_index(drop=True)

train_data = []
test_data = []

# Group by user to avoid cold-start in training
for user, group in triplet_df.groupby("user_id"):
    interactions = group.values.tolist()
    
    if len(interactions) == 1:
        # Keep single interaction in train
        train_data.extend(interactions)
    else:
        split_idx = int(0.8 * len(interactions))
        
        train_data.extend(interactions[:split_idx])
        test_data.extend(interactions[split_idx:])

# Convert back to DataFrame
train_df = pd.DataFrame(train_data, columns=["user_id", "item_id", "rating"])
test_df = pd.DataFrame(test_data, columns=["user_id", "item_id", "rating"])

# -----------------------------
# CHECKS
# -----------------------------
print("Train size:", len(train_df))
print("Test size:", len(test_df))

# Ensure all users exist in training
print("Unique users in train:", train_df["user_id"].nunique())
print("Unique users in test:", test_df["user_id"].nunique())

In [ ]:
train_df

In [ ]:
train_df['user_id'].nunique()

In [ ]:
train_df['item_id'].nunique()

In [ ]:
test_df

In [ ]:
import numpy as np
import pandas as pd

def generate_recommender_data(
    num_users=200_000,
    num_items=1000,
    interactions_per_user=20,
    test_ratio=0.2,
    seed=42
):
    np.random.seed(seed)

    # -----------------------------
    # STEP 1: Create latent factors (ground truth)
    # -----------------------------
    k_true = 10  # true latent features

    true_user_factors = np.random.normal(0, 1, (num_users, k_true))
    true_item_factors = np.random.normal(0, 1, (num_items, k_true))

    # Bias terms (important for realism)
    user_bias = np.random.normal(0, 0.5, num_users)
    item_bias = np.random.normal(0, 0.5, num_items)
    global_bias = 3.5

    # -----------------------------
    # STEP 2: Generate interactions
    # -----------------------------
    data = []

    for u in range(num_users):
        items = np.random.choice(num_items, interactions_per_user, replace=False)

        for i in items:
            rating = (
                global_bias
                + user_bias[u]
                + item_bias[i]
                + np.dot(true_user_factors[u], true_item_factors[i])
                + np.random.normal(0, 0.5)  # noise
            )

            # Clip ratings to 1–5
            rating = min(5, max(1, rating))

            data.append((u, i, rating))

    df = pd.DataFrame(data, columns=["user_id", "item_id", "rating"])

    # -----------------------------
    # STEP 3: Train-Test Split (user-wise)
    # -----------------------------
    train_data = []
    test_data = []

    for user, group in df.groupby("user_id"):
        group = group.sample(frac=1, random_state=seed)

        split_idx = int((1 - test_ratio) * len(group))

        train_data.append(group.iloc[:split_idx])
        test_data.append(group.iloc[split_idx:])

    train_df = pd.concat(train_data).reset_index(drop=True)
    test_df = pd.concat(test_data).reset_index(drop=True)

    return train_df, test_df

In [ ]:
train_df, test_df = generate_recommender_data()

In [ ]:
train_df

In [ ]:
test_df

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# -----------------------------
# MODEL
# -----------------------------
class MF(nn.Module):
    def __init__(self, num_users, num_items, k):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, k)
        self.item_emb = nn.Embedding(num_items, k)

        # Initialize small values
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def forward(self, users, items):
        u = self.user_emb(users)
        i = self.item_emb(items)
        return (u * i).sum(dim=1)  # dot product


# -----------------------------
# TRAIN FUNCTION
# -----------------------------
def train_mf_pytorch(train_df, test_df, unseen_df, k=15, lr=0.01, epochs=20, batch_size=4096):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # -----------------------------
    # MAPPINGS
    # -----------------------------
    user_map = {u: idx for idx, u in enumerate(train_df['user_id'].unique())}
    item_map = {i: idx for idx, i in enumerate(train_df['item_id'].unique())}

    num_users = len(user_map)
    num_items = len(item_map)

    # -----------------------------
    # DATA → TENSORS
    # -----------------------------
    user_ids = torch.tensor(train_df['user_id'].map(user_map).values, dtype=torch.long)
    item_ids = torch.tensor(train_df['item_id'].map(item_map).values, dtype=torch.long)
    ratings = torch.tensor(train_df['rating'].values, dtype=torch.float32)

    dataset = torch.utils.data.TensorDataset(user_ids, item_ids, ratings)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # -----------------------------
    # MODEL
    # -----------------------------
    model = MF(num_users, num_items, k).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    # -----------------------------
    # TRAINING
    # -----------------------------
    for epoch in range(1, epochs + 1):
        total_loss = 0

        for u, i, r in loader:
            u, i, r = u.to(device), i.to(device), r.to(device)

            preds = model(u, i)
            loss = loss_fn(preds, r)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch}/{epochs}, Loss: {total_loss:.4f}")

    # -----------------------------
    # RMSE
    # -----------------------------
    squared_error = 0
    count = 0

    model.eval()
    with torch.no_grad():
        for _, row in test_df.iterrows():
            u_raw, i_raw, r = row['user_id'], row['item_id'], row['rating']

            if u_raw not in user_map or i_raw not in item_map:
                continue

            u = torch.tensor([user_map[u_raw]], dtype=torch.long).to(device)
            i = torch.tensor([item_map[i_raw]], dtype=torch.long).to(device)

            pred = model(u, i).item()
            pred = np.clip(pred, 1, 5)

            squared_error += (r - pred) ** 2
            count += 1

    rmse = np.sqrt(squared_error / count)
    print(f"\nRMSE: {rmse:.4f}")

    # -----------------------------
    # RECOMMENDATIONS
    # -----------------------------
    recommendations = {}

    with torch.no_grad():
        for u_raw, group in unseen_df.groupby('user_id'):

            if u_raw not in user_map:
                continue

            u_idx = user_map[u_raw]
            u_tensor = torch.tensor([u_idx], dtype=torch.long).to(device)

            user_recs = []

            for _, row in group.iterrows():
                i_raw = row['item_id']

                if i_raw not in item_map:
                    continue

                i_idx = item_map[i_raw]
                i_tensor = torch.tensor([i_idx], dtype=torch.long).to(device)

                pred = model(u_tensor, i_tensor).item()
                pred = np.clip(pred, 1, 5)

                user_recs.append((i_raw, pred))

            user_recs.sort(key=lambda x: x[1], reverse=True)
            recommendations[u_raw] = user_recs[:5]

    return rmse, recommendations

In [ ]:
# -----------------------------
# CREATE UNSEEN DATA (10 USERS)
# -----------------------------

# Get 10 users from training data
selected_users = train_df['user_id'].unique()[:10]

# All items
all_items = set(train_df['item_id'].unique())

# User → seen items
user_seen = train_df.groupby('user_id')['item_id'].apply(set).to_dict()

data = []

for u in selected_users:
    seen_items = user_seen.get(u, set())
    
    # Get unseen items
    unseen_items = list(all_items - seen_items)
    
    # Take 20 unseen items per user (you can change this)
    sampled_items = unseen_items[:20]
    
    for i in sampled_items:
        data.append((u, i))

# Create DataFrame
unseen_df = pd.DataFrame(data, columns=["user_id", "item_id"])

print(unseen_df.head())
print("Total rows:", len(unseen_df))

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
rmse, pred=train_mf_pytorch(train_df,test_df,unseen_df,epochs=15)

In [ ]:
pred.items()

In [ ]:
def print_recommendations(recommendations, top_k=5):
    
    for user, recs in list(recommendations.items())[:10]:  # limit users
        
        print(f"\nUser {user}")
        print("-" * 40)
        print(f"{'Rank':<6}{'Item ID':<10}{'Predicted Rating'}")
        
        for rank, (item, score) in enumerate(recs[:top_k], start=1):
            print(f"{rank:<6}{item:<10}{score:.3f}")

print_recommendations(pred)